# Exercise 8.01 - Plotting Poaching Density

**Student ID:** mc641790  
**Course:** CISC7204 Suggested Exercise 08  
**Practice:** Practice 01

This notebook follows the requested workflow step by step and keeps the
supplied data file in the same `Jupyter Notebooks` folder.

## 1. Load the supplied data

`geoplotlib.read_csv` returns a `DataAccessObject`, which is optimized
for plotting. Pandas is also used below for tabular inspection.

> **Environment note:** These visualizations open in a separate interactive
> window. `geoplotlib 0.3.2` is a legacy package and does not install directly
> into the current Python 3.14 environment. Use a Python 3.10 or 3.11
> Anaconda environment for this tutorial. In **Anaconda Prompt**, run:
>
> ```text
> conda create -n cisc7204-py311 python=3.11 -y
> conda activate cisc7204-py311
> python -m pip install --no-build-isolation "setuptools<58" "geoplotlib==0.3.2" "pyglet<2"
> ```
>
> Then select the `cisc7204-py311` kernel in JupyterLab, restart the kernel,
> and run the notebook from the top. Close each map window before running the
> next visualization cell.

In [1]:
from pathlib import Path

DATA_FILE = Path("poaching_points_cleaned.csv")
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"{DATA_FILE} was not found. Put this notebook in the same "
        "'Jupyter Notebooks' folder as the supplied data file."
    )
print("Using:", DATA_FILE.resolve())

Using: ../data/poaching_points_cleaned.csv


In [2]:
import geoplotlib
from geoplotlib.utils import read_csv

dataset = read_csv(str(DATA_FILE))
dataset

DataAccessObject(['id_report', 'date_report', 'description', 'created_date', 'lat', 'lon'] x 268)

In [3]:
geoplotlib.tiles_provider({
    "url": lambda zoom, xtile, ytile:
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Street_Map/MapServer/tile/%d/%d/%d.jpg"
        % (zoom, ytile, xtile),
    "tiles_dir": "arcgis_world_street",
    "attribution": "Esri World Street Map",
})

In [4]:
import pandas as pd

pd_dataset = pd.read_csv(DATA_FILE, skipinitialspace=True)
print("Shape:", pd_dataset.shape)
pd_dataset.head()

Shape: (268, 6)


,id_report,date_report,description,created_date,lat,lon
0,138,01/01/2005 12:00:00 AM,Poaching incident,2005/01/01 12:00:00 AM,-7.049359,34.841440
1,4,01/20/2005 12:00:00 AM,Poaching incident,2005/01/20 12:00:00 AM,-7.650840,34.480010
2,43,01/20/2005 12:00:00 AM,Poaching incident,2005/02/20 12:00:00 AM,-7.843202,34.005704
3,98,01/20/2005 12:00:00 AM,Poaching incident,2005/02/21 12:00:00 AM,-7.745846,33.948526
4,141,01/20/2005 12:00:00 AM,Poaching incident,2005/02/22 12:00:00 AM,-7.876673,33.690167


The supplied file has the required `lat` and `lon` columns. The pandas
DataFrame supports preprocessing and descriptive analysis, whereas the
geoplotlib object is a lightweight plotting data container.

## 2. Dot-density plot

In [5]:
geoplotlib.dot(dataset)
geoplotlib.show()

The dots reveal clusters of reported incidents that are difficult to recognize from latitude and longitude values alone.

## 3. Histogram density plot

In [6]:
geoplotlib.tiles_provider({
    "url": lambda zoom, xtile, ytile:
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Street_Map/MapServer/tile/%d/%d/%d.jpg"
        % (zoom, ytile, xtile),
    "tiles_dir": "arcgis_world_street",
    "attribution": "Esri World Street Map",
})

In [7]:
geoplotlib.hist(dataset, binsize=20)
geoplotlib.show()

The histogram aggregates nearby points into cells, making hotspots and areas with few or no reports easier to compare.

## 4. Voronoi density plot

In [8]:
from pathlib import Path
from PIL import Image

blank_tile = Path("blank_tile.png")

if not blank_tile.exists():
    Image.new("RGB", (256, 256), (245, 245, 245)).save(blank_tile)

geoplotlib.tiles_provider({
    "url": lambda zoom, xtile, ytile: blank_tile.resolve().as_uri(),
    "tiles_dir": "offline_blank",
    "attribution": "Offline blank basemap",
})

In [9]:
geoplotlib.voronoi(dataset, cmap='Blues_r', max_area=1e5, alpha=255)
geoplotlib.show()

The Voronoi layer emphasizes the spatial influence around each report. Smaller, denser cells indicate areas where incidents are close together.